In [1]:
import pandas as pd
import sqlite3
import os

In [2]:
os.makedirs('../database', exist_ok=True)
print("Database folder ready")

Database folder ready


In [3]:
conn= sqlite3.connect('../database/ubereats.db')
print("connected to database!")

connected to database!


In [4]:
df = pd.read_csv('../data/clean/restaurants_clean.csv')
print(f"Loaded {len(df)} restaurants from CSV")
df.to_sql('restaurants', conn, if_exists='replace', index=False)
print('Restaurants table created in SQLite!')

Loaded 23012 restaurants from CSV
Restaurants table created in SQLite!


In [5]:
odf = pd.read_csv('../data/clean/jsonorders_clean.csv')
print(f"Loaded {len(odf)} orders from CSV")
odf.to_sql('orders', conn, if_exists='replace', index=False)
print('orders table created in SQLite!')

Loaded 25000 orders from CSV
orders table created in SQLite!


In [6]:
r_count = pd.read_sql_query("SELECT COUNT(*) FROM restaurants", conn).iloc[0,0]
o_count = pd.read_sql_query("SELECT COUNT(*) FROM orders", conn).iloc[0,0]
print(f"Restaurants table: {r_count} rows")
print(f"orders table: {o_count} rows")

Restaurants table: 23012 rows
orders table: 25000 rows


In [7]:
odf = pd.read_csv('../data/clean/jsonorders_clean.csv')
print(f"Orders CSV rows: {len(odf)}")
print(f"\nColumns: {odf.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(odf.head(3))

Orders CSV rows: 25000

Columns: ['order_id', 'restaurant_name', 'order_date', 'order_value', 'discount_used', 'payment_method', 'year', 'month', 'month_name']

First 3 rows:
                               order_id          restaurant_name  order_date  \
0  8174c2af-07a4-4837-9068-1169d963e36e           TBC Sky Lounge  2025-05-06   
1  0f9ddb57-4632-4a9f-9afe-1d2e41b94e32  KC Das - Sweet Paradise  2026-01-08   
2  099deda6-8c53-41cb-abf8-00126e6b2643        Hotel Kadamba Veg  2026-01-22   

   order_value discount_used payment_method  year  month month_name  
0       201.87            No           Card  2025      5        May  
1      1392.27            No           Cash  2026      1    January  
2      1358.35           Yes            UPI  2026      1    January  


In [8]:
print("--- Restaurants Sample ---")
print(pd.read_sql_query("SELECT * FROM restaurants LIMIT 3", conn))
print("\n--- Orders Sample ---")
print(pd.read_sql_query("SELECT * FROM orders LIMIT 3", conn))

--- Restaurants Sample ---
              name online_order book_table  rate  votes  \
0            Jalsa          Yes        Yes   4.1    775   
1   Spice Elephant          Yes         No   4.1    787   
2  San Churro Cafe          Yes         No   3.8    918   

                            phone      location            rest_type  \
0  080 42297555\r\n+91 9743772233  Banashankari        Casual Dining   
1                    080 41714161  Banashankari        Casual Dining   
2                  +91 9663487993  Banashankari  Cafe, Casual Dining   

                                          dish_liked  \
0  Pasta, Lunch Buffet, Masala Papad, Paneer Laja...   
1  Momos, Lunch Buffet, Chocolate Nirvana, Thai G...   
2  Churros, Cannelloni, Minestrone Soup, Hot Choc...   

                         cuisines  cost listed_type   listed_city  \
0  North Indian, Mughlai, Chinese   800      Buffet  Banashankari   
1     Chinese, North Indian, Thai   800      Buffet  Banashankari   
2          Cafe

Restaurant Analysis (10 Queries)

In [9]:
q1="""SELECT location,ROUND(AVG(rate),2) AS avg_rating,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY location
HAVING COUNT(*) > 10
ORDER BY avg_rating DESC
LIMIT 10;"""
result = pd.read_sql_query(q1, conn)
result

,location,avg_rating,total_restaurants
0,Lavelle Road,4.19,442
1,Koramangala 5th Block,4.15,1759
2,Sankey Road,4.11,17
3,St. Marks Road,4.10,304
4,Koramangala 3rd Block,4.10,161
5,Cunningham Road,4.10,333
6,Koramangala 2nd Block,4.07,45
7,Sadashiv Nagar,4.06,38
8,Residency Road,4.05,440
9,Church Street,4.05,506


In [10]:
q2 = """SELECT location, count(*) AS total_restaurants
FROM restaurants
GROUP BY location
ORDER BY total_restaurants DESC
LIMIT 10;"""
result = pd.read_sql_query(q2,conn)
result

,location,total_restaurants
0,Koramangala 5th Block,1759
1,BTM,1444
2,Indiranagar,1331
3,HSR,1154
4,Jayanagar,1030
5,JP Nagar,989
6,Whitefield,821
7,Koramangala 6th Block,718
8,Koramangala 7th Block,715
9,Marathahalli,678


In [11]:
q3 = """SELECT online_order,ROUND(AVG(rate),2) AS avg_rating,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY online_order;"""
result = pd.read_sql_query(q3, conn)
result

,online_order,avg_rating,total_restaurants
0,No,3.93,6738
1,Yes,3.89,16274


In [12]:
q4 = """SELECT book_table,ROUND(AVG(rate), 2) AS avg_rating,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY book_table;
"""
result = pd.read_sql_query(q4, conn)
result

,book_table,avg_rating,total_restaurants
0,No,3.81,16991
1,Yes,4.16,6021


In [13]:
q5="""SELECT price_segment,ROUND(AVG(rate),2) AS avg_rating,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY price_segment
ORDER BY avg_rating DESC;"""
result = pd.read_sql_query(q5,conn)
result

,price_segment,avg_rating,total_restaurants
0,Premium,4.07,8870
1,Low,3.83,3749
2,Medium,3.79,10393


In [14]:
q6="""SELECT price_segment,ROUND(AVG(rate),2) AS avg_rating,ROUND(AVG(cost),2) AS avg_cost,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY price_segment;"""
result=pd.read_sql_query(q6,conn)
result

,price_segment,avg_rating,avg_cost,total_restaurants
0,Low,3.83,245.64,3749
1,Medium,3.79,519.16,10393
2,Premium,4.07,1243.26,8870


In [15]:
q7="""SELECT cuisines,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY cuisines
ORDER BY total_restaurants DESC
LIMIT 10;"""
result = pd.read_sql_query(q7,conn)
result

,cuisines,total_restaurants
0,North Indian,1136
1,"North Indian, Chinese",776
2,South Indian,359
3,Cafe,273
4,"South Indian, North Indian, Chinese",233
5,"Bakery, Desserts",215
6,"Desserts, Beverages",214
7,Chinese,210
8,"Ice Cream, Desserts",208
9,Desserts,204


In [16]:
q8 ="""SELECT cuisines,ROUND(AVG(rate),2) AS avg_rating,COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY cuisines
HAVING COUNT(*)>20
ORDER BY avg_rating DESC
LIMIT 10;"""
result = pd.read_sql_query(q8,conn)
result

,cuisines,avg_rating,total_restaurants
0,"Continental, Asian, North Indian",4.72,21
1,"North Indian, Thai, Japanese, Continental, Cafe",4.69,34
2,"Cafe, American, Burger, Steak",4.60,43
3,"American, North Indian, Chinese, Finger Food",4.60,24
4,"North Indian, European, Mediterranean, BBQ, Kebab",4.53,39
5,"Italian, Pizza, Salad",4.50,28
6,"European, Continental",4.49,23
7,"Cafe, Desserts, Continental",4.49,24
8,"Ice Cream, Cafe, Pizza, Burger, Desserts, Beve...",4.42,33
9,"Pizza, Cafe, Italian",4.41,85


In [17]:
q9 ="""
SELECT CASE 
WHEN cost < 300 THEN 'under 300'
WHEN cost BETWEEN 300 AND 700 THEN '300-700'
ELSE 'Above 700'
END AS cost_bucket,
ROUND(AVG(rate),2) AS avg_rating,
COUNT(*) AS total_restaurants
FROM restaurants
GROUP BY cost_bucket;
"""
result =  pd.read_sql_query(q9, conn)
result

,cost_bucket,avg_rating,total_restaurants
0,300-700,3.79,12073
1,Above 700,4.07,8870
2,under 300,3.84,2069


In [18]:
q10="""SELECT location,
              COUNT(*) AS demand,
              ROUND(AVG(rate),2) AS avg_rating
FROM restaurants
GROUP BY location
HAVING COUNT(*)>50 AND AVG(rate)<4.0
ORDER BY demand DESC
LIMIT 5;
"""
result = pd.read_sql_query(q10, conn)
result

,location,demand,avg_rating
0,BTM,1444,3.76
1,Indiranagar,1331,3.96
2,HSR,1154,3.84
3,Jayanagar,1030,3.94
4,JP Nagar,989,3.85


JSON ORDER INSIGHTS

Order Analytics (5 queries)

    oq1 Which restaurant generates the highest revenue 

In [19]:
oq1="""
SELECT restaurant_name,
              ROUND(SUM(order_value),2) AS total_revenue,
              COUNT(*) AS total_orders
FROM orders
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10;
"""
result = pd.read_sql_query(oq1, conn)
result

,restaurant_name,total_revenue,total_orders
0,Bob's Bar,19518.14,15
1,Cake Cafe,19335.17,19
2,Biryani Mane,19249.45,15
3,Hungry Lee,19167.53,18
4,Andhra Grills,19153.72,19
5,Mighty Paws,18410.50,17
6,Delhi Ke Bawarchi,18131.68,15
7,Pingara,17954.06,15
8,Bawarchi Inn,17717.43,15
9,Soup'ermanz Kitchen,17350.72,13


oq2 Do discounts drive higher order values or increase order frequency 

In [20]:
oq2="""
SELECT discount_used,
       ROUND(AVG(order_value),2) AS avg_order_value,
       COUNT(*) AS total_orders,
       ROUND(SUM(order_value),2) AS total_revenue
FROM orders
GROUP BY discount_used;
"""
result = pd.read_sql_query(oq2, conn)
result

,discount_used,avg_order_value,total_orders,total_revenue
0,No,822.49,12509,10288535.97
1,Yes,1149.91,12491,14363510.85


oq3 which payment method is most popular

In [21]:
oq3 ="""
SELECT payment_method,
       COUNT(*) AS total_orders,
       ROUND(SUM(order_value),2) AS total_revenue,
       ROUND(AVG(order_value),2) AS avg_order_value
FROM orders
GROUP BY payment_method
ORDER BY total_orders DESC;
"""
result = pd.read_sql_query(oq3, conn)
result

,payment_method,total_orders,total_revenue,avg_order_value
0,Cash,8384,8245426.15,983.47
1,Card,8364,8272199.32,989.02
2,UPI,8252,8134421.35,985.75


oq4 how does the monthly revenue trend look like

In [22]:
oq4 = """
SELECT year, month, month_name,
       COUNT(*) AS orders,
       ROUND(SUM(order_value), 2) AS revenue,
       ROUND(AVG(order_value), 2) AS avg_order
FROM orders
GROUP BY year, month
ORDER BY year, month;
"""

result = pd.read_sql_query(oq4, conn)
result

,year,month,month_name,orders,revenue,avg_order
0,2025,4,April,152,150980.60,993.29
1,2025,5,May,2592,2573888.68,993.01
2,2025,6,June,2422,2374143.00,980.24
3,2025,7,July,2613,2567747.54,982.68
4,2025,8,August,2520,2444881.74,970.19
5,2025,9,September,2483,2449075.72,986.34
6,2025,10,October,2531,2527196.71,998.50
7,2025,11,November,2478,2451802.43,989.43
8,2025,12,December,2664,2580580.57,968.69
9,2026,1,January,2642,2630759.98,995.75


oq5 which restaurants have most repeated orders

In [23]:
oq5 =""" 
SELECT restaurant_name,
       COUNT(*) AS repeat_orders,
       ROUND(AVG(order_value),2) AS avg_order_value,
       ROUND(SUM(order_value),2) AS total_revenue
FROM orders
GROUP BY restaurant_name
ORDER BY repeat_orders DESC
LIMIT 10;
"""
result = pd.read_sql_query(oq5, conn)
result

,restaurant_name,repeat_orders,avg_order_value,total_revenue
0,Cake Cafe,19,1017.64,19335.17
1,Andhra Grills,19,1008.09,19153.72
2,Shree Muthahalli Veg,18,842.29,15161.28
3,Reddy's Restaurant,18,928.25,16708.49
4,Orbis Restaurant,18,884.54,15921.81
5,Hungry Lee,18,1064.86,19167.53
6,Mighty Paws,17,1082.97,18410.50
7,Melt,17,942.11,16015.88
8,JW Kitchen - JW Marriott Bengaluru,17,813.56,13830.45
9,Aubree,17,973.72,16553.31
